# Churn Definition and Label Construction

## 0. Load Data

This notebook defines the time-aware churn prediction framework and constructs the future churn label using rolling customer snapshots.

The framework uses:

- a 60-day observation window;
- a 30-day prediction horizon;
- eligibility criteria requiring meaningful prior activity;
- future behavioral deterioration to define churn;
- strict temporal separation between predictors and outcomes.

No future-window variables constructed in this notebook will be used as model features.


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path(
    "/Users/qiaodou/Desktop/onemoreclass-churn-intelligence"
)

os.chdir(PROJECT_ROOT)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

customers = pd.read_csv(
    RAW_DATA_DIR / "customers.csv",
    parse_dates=["signup_date"]
)

activities = pd.read_csv(
    RAW_DATA_DIR / "activities.csv",
    parse_dates=["activity_date"]
)

print("customers:", customers.shape)
print("activities:", activities.shape)

print(
    "\nActivity date range:",
    activities["activity_date"].min(),
    "to",
    activities["activity_date"].max()
)


## 1. Snapshot Framework

Each modeling row represents one customer at one snapshot date.

For each snapshot:

- Features are calculated using the prior 60 days.
- Churn is determined using the following 30 days.
- Only information available on or before the snapshot date may enter the feature set.


### 1.1 Generate Snapshot Dates


In [ ]:
OBSERVATION_DAYS = 60
PREDICTION_DAYS = 30
SNAPSHOT_STEP_DAYS = 14

activity_start = activities["activity_date"].min()
activity_end = activities["activity_date"].max()

first_snapshot = (
    activity_start
    + pd.Timedelta(days=OBSERVATION_DAYS - 1)
)

last_snapshot = (
    activity_end
    - pd.Timedelta(days=PREDICTION_DAYS)
)

snapshot_dates = pd.date_range(
    start=first_snapshot,
    end=last_snapshot,
    freq=f"{SNAPSHOT_STEP_DAYS}D"
)

print("First snapshot:", first_snapshot.date())
print("Last possible snapshot:", last_snapshot.date())
print("Number of snapshots:", len(snapshot_dates))

print("\nSnapshot dates:")

for date in snapshot_dates:
    print(date.date())


### 1.2 Define build_snapshot_labels()


In [ ]:
def build_snapshot_labels(
    customers: pd.DataFrame,
    activities: pd.DataFrame,
    snapshot_date: pd.Timestamp,
    observation_days: int = 60,
    prediction_days: int = 30
) -> pd.DataFrame:

    observation_start = (
        snapshot_date
        - pd.Timedelta(days=observation_days - 1)
    )

    future_start = snapshot_date + pd.Timedelta(days=1)
    future_end = snapshot_date + pd.Timedelta(days=prediction_days)

    past = activities[
        activities["activity_date"].between(observation_start, snapshot_date)
    ].copy()

    past["active_login_day"] = (past["login_count"] > 0).astype(int)

    past_summary = (
        past.groupby("customer_id").agg(
            past_active_days=("active_login_day", "sum"),
            past_logins=("login_count", "sum"),
            past_minutes=("minutes_watched", "sum"),
            past_assignments=("assignments_completed", "sum")
        ).reset_index()
    )

    future = activities[
        activities["activity_date"].between(future_start, future_end)
    ].copy()

    future["active_login_day"] = (future["login_count"] > 0).astype(int)

    future_summary = (
        future.groupby("customer_id").agg(
            future_active_days=("active_login_day", "sum"),
            future_logins=("login_count", "sum"),
            future_minutes=("minutes_watched", "sum"),
            future_assignments=("assignments_completed", "sum")
        ).reset_index()
    )

    snapshot = (
        customers[["customer_id"]]
        .merge(past_summary, on="customer_id", how="left")
        .merge(future_summary, on="customer_id", how="left")
    )

    activity_cols = [
        "past_active_days",
        "past_logins",
        "past_minutes",
        "past_assignments",
        "future_active_days",
        "future_logins",
        "future_minutes",
        "future_assignments",
    ]

    snapshot[activity_cols] = snapshot[activity_cols].fillna(0)
    snapshot["snapshot_date"] = snapshot_date

    return snapshot


## 2. Single-Snapshot Validation


### 2.1 Create Test Snapshot


In [ ]:
test_snapshot = build_snapshot_labels(
    customers=customers,
    activities=activities,
    snapshot_date=snapshot_dates[0]
)

print(test_snapshot.head())
print("\nShape:", test_snapshot.shape)

summary_cols = [
    "past_active_days",
    "past_logins",
    "past_minutes",
    "past_assignments",
    "future_active_days",
    "future_logins",
    "future_minutes",
    "future_assignments"
]

print("\nSnapshot activity summary:")
print(
    test_snapshot[summary_cols]
    .describe()
    .round(2)
)

### 2.2 Inspect Snapshot Completeness

This step performs a basic sanity check to confirm that the snapshot contains one observation per customer and that the required past and future activity variables are populated correctly.

In [ ]:
print(
    "Unique customers:",
    test_snapshot["customer_id"].nunique()
)

print(
    "Total rows:",
    len(test_snapshot)
)

print(
    "\nMissing values:"
)

print(
    test_snapshot[
        [
            "past_active_days",
            "past_logins",
            "past_minutes",
            "past_assignments",
            "future_active_days",
            "future_logins",
            "future_minutes",
            "future_assignments"
        ]
    ]
    .isna()
    .sum()
)

### 2.3 Define Eligible Modeling Population

Customers must have sufficient prior engagement for future behavioral decline to be meaningfully measured.

A customer-snapshot is eligible if the customer had:
- at least 15 active login days during the prior 60 days; and
- at least 500 watched minutes during the prior 60 days.

Use constants:

```python
MIN_ACTIVE_DAYS = 15
MIN_WATCH_MINUTES = 500
```


In [ ]:
MIN_ACTIVE_DAYS = 15
MIN_WATCH_MINUTES = 500

test_snapshot["eligible"] = (
    (test_snapshot["past_active_days"] >= MIN_ACTIVE_DAYS)
    & (test_snapshot["past_minutes"] >= MIN_WATCH_MINUTES)
).astype(int)

print("\nEligibility:")
print(test_snapshot["eligible"].value_counts().sort_index())
print("\nEligibility rate:", round(test_snapshot["eligible"].mean(), 3))


## 3. Initial Hard-Churn Definition — Rejected

An initial definition based on near-complete future inactivity was tested first.

A customer would be considered churned if, during the next 30 days, they had:

- no more than 2 active login days;
- fewer than 60 watched minutes;
- no more than 1 completed assignment.

This definition produced virtually no positive cases and was therefore rejected as too restrictive for an educational platform where churn is more likely to emerge as substantial disengagement rather than immediate complete inactivity.


In [ ]:
test_snapshot["hard_churn"] = (
    (test_snapshot["future_active_days"] <= 2)
    & (test_snapshot["future_minutes"] < 60)
    & (test_snapshot["future_assignments"] <= 1)
).astype(int)

eligible_snapshot = test_snapshot[test_snapshot["eligible"] == 1].copy()

print("Eligible customers:", len(eligible_snapshot))
print("Hard churners:", eligible_snapshot["hard_churn"].sum())
print("Hard churn rate:", round(eligible_snapshot["hard_churn"].mean(), 4))


## 4. Behavioral Churn Calibration

The initial churn definition was too restrictive for the observed activity distribution and classified no customers as churners.

Rather than selecting arbitrary thresholds, the future activity distribution is examined to define meaningful disengagement relative to normal customer behavior.


### 4.1 Future Activity Distribution


In [ ]:
future_cols = [
    "future_active_days",
    "future_logins",
    "future_minutes",
    "future_assignments"
]

print(
    test_snapshot[future_cols]
    .describe(
        percentiles=[
            0.05,
            0.10,
            0.15,
            0.20,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
    .round(2)
)

for col in [
    "future_active_days",
    "future_minutes",
    "future_assignments"
]:
    print(f"\n=== {col} ===")

    print(
        test_snapshot[col]
        .quantile([
            0.01,
            0.05,
            0.10,
            0.15,
            0.20,
            0.25,
            0.50
        ])
        .round(2)
    )

### 4.2 Normalize Observation and Prediction Windows

Because the observation window contains 60 days while the prediction window contains 30 days, raw totals are not directly comparable. Activity metrics are converted to per-day rates before calculating future behavioral deterioration.


In [ ]:
test_snapshot["past_active_day_rate"] = (
    test_snapshot["past_active_days"]
    / OBSERVATION_DAYS
)

test_snapshot["future_active_day_rate"] = (
    test_snapshot["future_active_days"]
    / PREDICTION_DAYS
)

test_snapshot["past_minutes_per_day"] = (
    test_snapshot["past_minutes"]
    / OBSERVATION_DAYS
)

test_snapshot["future_minutes_per_day"] = (
    test_snapshot["future_minutes"]
    / PREDICTION_DAYS
)

test_snapshot["past_assignments_per_day"] = (
    test_snapshot["past_assignments"]
    / OBSERVATION_DAYS
)

test_snapshot["future_assignments_per_day"] = (
    test_snapshot["future_assignments"]
    / PREDICTION_DAYS
)

### 4.3 Zero-Safe Decline Rates

A direct percentage-change calculation can become unstable when past activity is zero. Therefore, decline rates are calculated using a zero-safe transformation.

Do not use epsilon-based decline calculations.

In [ ]:
def safe_decline_rate(past, future):
    decline = np.where(
        past > 0,
        (past - future) / past,
        np.where(
            future == 0,
            0.0,
            -1.0
        )
    )

    return decline


test_snapshot["future_login_decline"] = (
    safe_decline_rate(
        test_snapshot["past_active_day_rate"],
        test_snapshot["future_active_day_rate"]
    )
)

test_snapshot["future_minutes_decline"] = (
    safe_decline_rate(
        test_snapshot["past_minutes_per_day"],
        test_snapshot["future_minutes_per_day"]
    )
)

test_snapshot["future_assignment_decline"] = (
    safe_decline_rate(
        test_snapshot["past_assignments_per_day"],
        test_snapshot["future_assignments_per_day"]
    )
)


decline_cols = [
    "future_login_decline",
    "future_minutes_decline",
    "future_assignment_decline"
]

print(
    test_snapshot[decline_cols]
    .describe(
        percentiles=[
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.80,
            0.85,
            0.90,
            0.95
        ]
    )
    .round(3)
)

### 4.4 Threshold Sensitivity Analysis


In [ ]:
threshold_scenarios = {
    "strict": {"login": 0.15, "minutes": 0.25, "assignments": 0.40},
    "moderate": {"login": 0.10, "minutes": 0.20, "assignments": 0.30},
    "sensitive": {"login": 0.08, "minutes": 0.15, "assignments": 0.25},
}

results = []

for scenario, thresholds in threshold_scenarios.items():
    signal_count = (
        (test_snapshot["future_login_decline"] >= thresholds["login"]).astype(int)
        + (test_snapshot["future_minutes_decline"] >= thresholds["minutes"]).astype(int)
        + (test_snapshot["future_assignment_decline"] >= thresholds["assignments"]).astype(int)
    )

    churn = (signal_count >= 2) & (test_snapshot["eligible"] == 1)

    results.append({
        "scenario": scenario,
        "login_threshold": thresholds["login"],
        "minutes_threshold": thresholds["minutes"],
        "assignment_threshold": thresholds["assignments"],
        "churn_count": churn.sum(),
        "churn_rate": churn.sum() / test_snapshot["eligible"].sum(),
    })

threshold_results = pd.DataFrame(results)

print(
    threshold_results[[
        "scenario",
        "login_threshold",
        "minutes_threshold",
        "assignment_threshold",
        "churn_count",
        "churn_rate"
    ]].round(3)
)


### 4.5 Final Threshold Selection

The moderate definition was selected because it captures meaningful multidimensional deterioration without restricting the positive class to only the most extreme cases.

A churn label requires at least two of the following future changes:

- login activity decline ≥ 10%;
- watched-minutes decline ≥ 20%;
- assignment activity decline ≥ 30%.

These thresholds define the outcome and are not model hyperparameters.


In [ ]:
LOGIN_DECLINE_THRESHOLD = 0.10
MINUTES_DECLINE_THRESHOLD = 0.20
ASSIGNMENT_DECLINE_THRESHOLD = 0.30

## 5. Build Full Customer-Snapshot Panel


In [ ]:
snapshot_list = []

for snapshot_date in snapshot_dates:
    snapshot = build_snapshot_labels(
        customers=customers,
        activities=activities,
        snapshot_date=snapshot_date,
        observation_days=OBSERVATION_DAYS,
        prediction_days=PREDICTION_DAYS
    )
    snapshot_list.append(snapshot)

snapshot_panel = pd.concat(snapshot_list, ignore_index=True)

print("Snapshot panel shape:", snapshot_panel.shape)
print("\nUnique snapshot dates:", snapshot_panel["snapshot_date"].nunique())
print("\nRows by snapshot:")
print(snapshot_panel.groupby("snapshot_date").size())


## 6. Apply Eligibility and Construct Outcome Variables

This section applies the eligibility criteria and constructs the normalized behavioral outcome variables required for the final churn definition.


In [ ]:
snapshot_panel["eligible"] = (
    (
        snapshot_panel["past_active_days"]
        >= MIN_ACTIVE_DAYS
    )
    &
    (
        snapshot_panel["past_minutes"]
        >= MIN_WATCH_MINUTES
    )
).astype(int)


snapshot_panel["past_active_day_rate"] = (
    snapshot_panel["past_active_days"]
    / OBSERVATION_DAYS
)

snapshot_panel["future_active_day_rate"] = (
    snapshot_panel["future_active_days"]
    / PREDICTION_DAYS
)

snapshot_panel["past_minutes_per_day"] = (
    snapshot_panel["past_minutes"]
    / OBSERVATION_DAYS
)

snapshot_panel["future_minutes_per_day"] = (
    snapshot_panel["future_minutes"]
    / PREDICTION_DAYS
)

snapshot_panel["past_assignments_per_day"] = (
    snapshot_panel["past_assignments"]
    / OBSERVATION_DAYS
)

snapshot_panel["future_assignments_per_day"] = (
    snapshot_panel["future_assignments"]
    / PREDICTION_DAYS
)


snapshot_panel["future_login_decline"] = (
    safe_decline_rate(
        snapshot_panel["past_active_day_rate"],
        snapshot_panel["future_active_day_rate"]
    )
)

snapshot_panel["future_minutes_decline"] = (
    safe_decline_rate(
        snapshot_panel["past_minutes_per_day"],
        snapshot_panel["future_minutes_per_day"]
    )
)

snapshot_panel["future_assignment_decline"] = (
    safe_decline_rate(
        snapshot_panel["past_assignments_per_day"],
        snapshot_panel["future_assignments_per_day"]
    )
)

## 7. Final Churn Definition


### 7.1 Generate Decline Signals

The decline signals are created from the normalized future behavior relative to the observation window.


In [ ]:
snapshot_panel["login_decline_flag"] = (
    snapshot_panel["future_login_decline"]
    >= LOGIN_DECLINE_THRESHOLD
).astype(int)

snapshot_panel["minutes_decline_flag"] = (
    snapshot_panel["future_minutes_decline"]
    >= MINUTES_DECLINE_THRESHOLD
).astype(int)

snapshot_panel["assignment_decline_flag"] = (
    snapshot_panel["future_assignment_decline"]
    >= ASSIGNMENT_DECLINE_THRESHOLD
).astype(int)

snapshot_panel["decline_signal_count"] = (
    snapshot_panel["login_decline_flag"]
    + snapshot_panel["minutes_decline_flag"]
    + snapshot_panel["assignment_decline_flag"]
)

### 7.2 Construct Churn Label

The final churn outcome is a 2-out-of-3 rule based on persistent future disengagement across the main activity dimensions.


In [ ]:
snapshot_panel["churn"] = np.where(
    snapshot_panel["eligible"] == 1,
    (
        snapshot_panel["decline_signal_count"]
        >= 2
    ).astype(int),
    np.nan
)

### 7.3 Create Final Modeling Population

Only eligible customer-snapshots are retained for downstream modeling.


In [ ]:
modeling_panel = (
    snapshot_panel[
        snapshot_panel["eligible"] == 1
    ]
    .copy()
)

modeling_panel["churn"] = (
    modeling_panel["churn"]
    .astype(int)
)

print(
    "Modeling panel shape:",
    modeling_panel.shape
)

print(
    "\nChurn distribution:"
)

print(
    modeling_panel["churn"]
    .value_counts()
    .sort_index()
)

print(
    "\nFinal churn rate:",
    round(
        modeling_panel["churn"].mean(),
        3
    )
)

## 8. Label Validation and Diagnostics


### 8.1 Churn Rate by Snapshot


In [ ]:
snapshot_validation = (
    modeling_panel
    .groupby("snapshot_date")
    .agg(
        eligible_customers=("customer_id", "count"),
        churners=("churn", "sum"),
        churn_rate=("churn", "mean")
    )
    .reset_index()
)

snapshot_validation["churn_rate"] *= 100

print("=== CHURN RATE BY SNAPSHOT ===")
print(snapshot_validation)

overall_churn_rate = modeling_panel["churn"].mean() * 100
print("\nOverall churn rate:", round(overall_churn_rate, 2), "%")


In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=snapshot_validation, x="snapshot_date", y="churn_rate", marker="o")
plt.axhline(snapshot_validation["churn_rate"].mean(), linestyle="--", label="Average")
plt.title("Churn Rate Across Snapshot Dates")
plt.xlabel("Snapshot Date")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


The churn rate increases across the earlier snapshots and stabilizes around 23% in the final observation periods.

This pattern suggests a temporal component to disengagement risk but no longer exhibits the continuously accelerating end-of-window artifact observed in the earlier synthetic-data design.


### 8.2 Signal Trends

This diagnostic reviews how the three decline signals evolve across snapshots and confirms that the moderate threshold definition remains stable over time.


In [ ]:
signal_by_snapshot = (
    modeling_panel
    .groupby("snapshot_date")
    .agg(
        avg_login_decline=("future_login_decline", "mean"),
        avg_minutes_decline=("future_minutes_decline", "mean"),
        avg_assignment_decline=("future_assignment_decline", "mean"),
        avg_signal_count=("decline_signal_count", "mean"),
        churn_rate=("churn", "mean")
    )
    .reset_index()
)

numeric_cols = signal_by_snapshot.columns.drop("snapshot_date")

signal_by_snapshot[numeric_cols] = (
    signal_by_snapshot[numeric_cols]
    .round(3)
)

print(signal_by_snapshot)

### 8.3 Signal Trigger Rates


In [ ]:
signal_trigger_rates = (
    modeling_panel
    .groupby("snapshot_date")
    .agg(
        login_decline_rate=("login_decline_flag", "mean"),
        minutes_decline_rate=("minutes_decline_flag", "mean"),
        assignment_decline_rate=("assignment_decline_flag", "mean")
    )
    .reset_index()
)

rate_cols = [
    "login_decline_rate",
    "minutes_decline_rate",
    "assignment_decline_rate"
]

signal_trigger_rates[rate_cols] = (
    signal_trigger_rates[rate_cols]
    * 100
)

signal_trigger_rates[rate_cols] = (
    signal_trigger_rates[rate_cols]
    .round(2)
)

print(signal_trigger_rates)

### 8.4 Churn Persistence


In [ ]:
churn_persistence = (
    modeling_panel
    .groupby("customer_id")
    .agg(
        snapshots_observed=("snapshot_date", "count"),
        churn_snapshots=("churn", "sum")
    )
    .reset_index()
)

churn_persistence["churn_snapshot_share"] = (
    churn_persistence["churn_snapshots"] / churn_persistence["snapshots_observed"]
)

print(churn_persistence["churn_snapshots"].value_counts().sort_index())
print("\nCustomers ever labeled churn:", (churn_persistence["churn_snapshots"] > 0).mean().round(3))


## 9. Modeling Guardrails

The following variables are outcome-construction variables and must never be used as predictors:

- future_active_days
- future_logins
- future_minutes
- future_assignments
- future_active_day_rate
- future_minutes_per_day
- future_assignments_per_day
- future_login_decline
- future_minutes_decline
- future_assignment_decline
- login_decline_flag
- minutes_decline_flag
- assignment_decline_flag
- decline_signal_count

Only information available on or before each snapshot date may be used in downstream feature engineering and churn modeling.

Because customers appear in multiple overlapping snapshots, downstream model validation must use temporal splits rather than random row-level train/test splits.


## 10. Save Modeling Population

The finalized eligible customer-snapshot population and churn labels are saved for downstream feature engineering.

In [ ]:
label_cols = [
    "customer_id",
    "snapshot_date",
    "churn"
]

modeling_labels = (
    modeling_panel[label_cols]
    .copy()
)

output_path = (
    PROCESSED_DATA_DIR
    / "modeling_labels.csv"
)

modeling_labels.to_csv(
    output_path,
    index=False
)

print(
    "Saved modeling labels:",
    output_path
)

print(
    "Shape:",
    modeling_labels.shape
)